In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

In [ ]:
df = yf.download("TSLA", start="2015-01-01", end="2025-01-01")
df.head()

In [ ]:
df.to_csv("data/tesla_stock_data.csv")

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nData Types:\n", df.dtypes)

In [ ]:
print(df.isnull().sum())


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
print("duplicate rows:", df.duplicated().sum())

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(df['Close'])
plt.title("Tesla Closing Price Trend")
plt.xlabel("Date")
plt.ylabel("Closing Price")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['Close'], bins=30, kde=True)

plt.title("Distribution of Closing Prices")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df[['Open', 'High', 'Low', 'Close']])
plt.title("Boxplot for Stock Features")

plt.show()

In [ ]:
sns.pairplot(df[['Open', 'High', 'Low', 'Close']])

plt.show()

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(df['Volume'])
plt.title("Trading Volume Trend")
plt.xlabel("Date")
plt.ylabel("Volume")

plt.show()

# EDA Insights

1. Tesla stock prices show significant fluctuations over time.

2. Open, High, Low, and Close prices are highly correlated.

3. The dataset contains very few missing values.

4. Some outliers are present due to market volatility.

5. Trading volume changes significantly during major market movements.

In [ ]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Display first 5 rows
df.head()

In [ ]:
# Daily return
df['Daily_Return'] = df['Close'].pct_change()

# 7-day moving average
df['MA7'] = df['Close'].rolling(window=7).mean()

# 30-day moving average
df['MA30'] = df['Close'].rolling(window=30).mean()

# Volatility
df['Volatility'] = df['Daily_Return'].rolling(window=7).std()

# Display dataset
df.head()

In [ ]:
# Missing values
print(df.isnull().sum())

In [ ]:
#Remove Missing values
df.dropna(inplace=True)

#Check shape
print(df.shape)

In [ ]:
print(df.columns)

In [ ]:
# Features

X = df[['Open', 'High', 'Low', 'Close', 'Volume', 'Daily_Return', 'MA7', 'MA30', 'Volatility']]

# Target

y = df['Target']
print(X.head())
print(y.head())

In [ ]:
# Split dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

In [ ]:
# Feature scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
import joblib

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# Logistic Regression

lr_model = LogisticRegression()

lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_pred = lr_model.predict(X_test_scaled)

# Accuracy
lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy:", lr_accuracy)

In [ ]:
# Random Forest

rf_model = RandomForestClassifier(n_estimators=200,max_depth=10, random_state=42 )

rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

# Accuracy
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

In [ ]:
# XGBoost Model

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)

# Accuracy
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print("XGBoost Accuracy:", xgb_accuracy)

In [ ]:
# Accuracy comparison

accuracy_df = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy
    ]
})

accuracy_df.sort_values(by='Accuracy', ascending=False, inplace=True)

In [ ]:
# Accuracy comparison graph

plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=accuracy_df
)

plt.title("Model Accuracy Comparison")

plt.ylim(0,1)

plt.show()

In [ ]:
# Find best model

best_accuracy = accuracy_df['Accuracy'].max()

best_model_name = accuracy_df.loc[
    accuracy_df['Accuracy'].idxmax(),
    'Model'
]

print("Best Model:", best_model_name)
print("Best Accuracy:", best_accuracy)

In [ ]:
# Classification report for XGBoost

print(classification_report(y_test, xgb_pred))

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# Save final model

joblib.dump(xgb_model, 'models/model.pkl')              

In [ ]:
# RSI Calculation

delta = df['Close'].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss

df['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
#EMA Calculation
df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()

df['EMA_30'] = df['Close'].ewm(span=30, adjust=False).mean()

In [ ]:
# MACD Feature

df['MACD'] = df['EMA_10'] - df['EMA_30']

In [ ]:
# Remove missing values again

df.dropna(inplace=True)
print(df.shape)

In [ ]:
# Improved features

X = df[[
    'Open',
    'High',
    'Low',
    'Close',
    'Volume',
    'Daily_Return',
    'MA7',
    'MA30',
    'Volatility',
    'RSI',
    'EMA_10',
    'EMA_30',
    'MACD'
]]

y = df['Target']

In [ ]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
# Optimized Random Forest

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Optimized Random Forest Accuracy:", rf_accuracy)

In [ ]:
# Optimized XGBoost

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)

print("Optimized XGBoost Accuracy:", xgb_accuracy)

In [ ]:
# New target based on moving average trend

df['Target'] = (df['MA7'] > df['MA30']).astype(int)

# Check target distribution
print(df['Target'].value_counts())

In [ ]:
# Features

X = df[[
    'Open',
    'High',
    'Low',
    'Close',
    'Volume',
    'Daily_Return',
    'Volatility',
    'RSI',
    'EMA_10',
    'EMA_30',
    'MACD'
]]

# Target

y = df['Target']

In [ ]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Random Forest

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

In [ ]:
# XGBoost

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_pred)

print("XGBoost Accuracy:", xgb_accuracy)

In [ ]:
# Final model comparison

accuracy_df = pd.DataFrame({
    'Model': [
        'Random Forest',
        'XGBoost'
    ],
    'Accuracy': [
        rf_accuracy,
        xgb_accuracy
    ]
})

accuracy_df.sort_values(by='Accuracy', ascending=False, inplace=True)

In [ ]:
# Accuracy graph

plt.figure(figsize=(8,5))

sns.barplot(
    x='Model',
    y='Accuracy',
    data=accuracy_df
)

plt.title("Model Accuracy Comparison")

plt.ylim(0.8, 1.0)

plt.show()

In [ ]:
# Classification Report

print(classification_report(y_test, rf_pred))

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
# Convert column names to simple strings

X.columns = [str(col) for col in X.columns]

print(X.columns)


In [ ]:
# Feature importance dataframe

feature_importance = pd.DataFrame()

feature_importance['Feature'] = X.columns

feature_importance['Importance'] = rf_model.feature_importances_

# Sort values

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

# Plot

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.xlabel("Importance")

plt.ylabel("Feature")

plt.title("Feature Importance")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
# Save final model

joblib.dump(rf_model, 'models/model.pkl')

print("Model saved successfully")

In [ ]:
# Check class balance

print(df['Target'].value_counts())

In [ ]:
# Balanced Random Forest

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Balanced Random Forest Accuracy:", rf_accuracy)

In [ ]:
joblib.dump(rf_model, 'models/model.pkl')